In [3]:
import nibabel as nib
import numpy as np
import os

def create_gm_wm_mask(aparc_path, aseg_path, out_path):
    """
    Extract Cortical GM and Cerebral WM masks from FastSurfer outputs.
    
    aparc_path: path to aparc.DKTatlas+aseg.deep.mgz
    aseg_path: path to aseg.auto_noCCseg.mgz
    out_path: where to save the final mask (NIfTI)
    """

    # Load segmentations
    aparc_img = nib.load(aparc_path)
    aseg_img = nib.load(aseg_path)
    aparc = aparc_img.get_fdata().astype(int)
    aseg = aseg_img.get_fdata().astype(int)

    # -----------------------------
    # Label definitions (from FreeSurferColorLUT)
    # -----------------------------

    # Cortical GM (aparc)
    cortical_gm_labels_aparc = list(range(1000, 1036)) + list(range(2000, 2036))

    # Cortical GM (aseg)
    cortical_gm_labels_aseg = [3, 42]

    # Cerebral WM (aseg main labels)
    wm_labels = [2, 41, 77, 78, 79, 219, 223]  # add WM hypointensities + WM edge if needed


    # CSF/Ventricles to remove
    csf_labels = [4, 5, 14, 15, 24, 31, 43, 44, 72]  
    
    # (includes lateral ventricles, 3rd, 4th, CSF, choroid plexus, etc.)

    # -----------------------------
    # Create masks
    # -----------------------------
    gm_mask_aparc = np.isin(aparc, cortical_gm_labels_aparc)
    gm_mask_aseg = np.isin(aseg, cortical_gm_labels_aseg)
    wm_mask = np.isin(aseg, wm_labels)
    csf_mask = np.isin(aseg, csf_labels)

    # Combine GM + WM, exclude CSF
    final_mask = np.zeros_like(aparc, dtype=np.uint8)
    final_mask[gm_mask_aparc] = 1   # GM = 1
    final_mask[gm_mask_aseg] = 1    # GM = 1
    final_mask[wm_mask] = 2   # WM = 2
    final_mask[csf_mask] = 0  # force CSF to background

    # -----------------------------
    # Save NIfTI
    # -----------------------------
    out_img = nib.Nifti1Image(final_mask, aparc_img.affine, aparc_img.header)
    nib.save(out_img, out_path)

    print(f"✅ Saved mask to {out_path}")

In [4]:
path = 'MRI_FASTSURFER_IMG_MASK/masks'
out = 'MRI_FASTSURFER_IMG_MASK/gm_wm_masks'

os.makedirs(out, exist_ok=True)
for f in os.listdir(path):
    if 'aparc' in f:
        aparc_file = os.path.join(path, f)
        aseg_file = aparc_file.replace('aparc', 'aseg')
        output_file = f.replace('aparc', 'gm_wm_mask')
        output_file = os.path.join(out, output_file)
        create_gm_wm_mask(aparc_file, aseg_file, output_file)
    else:
        continue

    


✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_126_S_0405_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20070209175521333_S15644_I38807_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_023_S_0604_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20080111140746085_S43623_I87077_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_002_S_0685_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20081015130623197_S55756_I120998_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_013_S_1275_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20070727184411020_S27071_I62674_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_094_S_1188_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20090204185159623_S62195_I135285_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_wm_masks/ADNI_011_S_0053_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20061206182608127_S15006_I31999_gm_wm_mask.nii.gz
✅ Saved mask to MRI_FASTSURFER_IMG_MASK/gm_w